# 02 · Dump frozen RGB features (resumable)
Runs CorrNet's frozen spatial frontend over every sample, saving one
`[T, 512]` fp16 array per id. **Resumable:** finished ids are skipped, so a
Colab disconnect costs only the current sample. **Gate:** all 6841 samples
dumped, failure log empty.

> Report wording: *frame-level features from a frozen sign-tuned
> spatio-temporal frontend* (CorrNet's corr-modules use neighbour frames).

In [ ]:
# --- Colab bootstrap (run first in every notebook) ---
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT = '/content/drive/MyDrive/cslr_phoenix/project'   # <- where this code lives
sys.path.append(PROJECT)
%cd $PROJECT

!pip -q install pyyaml
from src.utils import load_config
cfg = load_config('config.yaml')
print('config loaded:', cfg['project']['name'])


## ADAPT_HERE #1+#2 — load CorrNet frontend & set the frame transform
Clone + load weights. We grab the 2D backbone (`model.conv2d`) as the frozen
extractor. Confirm the input transform against `dataloader_video.py`.

In [ ]:
import torch, sys
repo = cfg['paths']['corrnet_repo']
sys.path.append(repo)
import os
if not os.path.exists(repo):
    !git clone https://github.com/hulianyuyy/CorrNet $repo

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# NOTE: class/import path may differ slightly by commit — adjust if needed.
from slr_network import SLRModel  # ADAPT_HERE if module name differs

ckpt = torch.load(cfg['paths']['corrnet_checkpoint'], map_location=device)
state = ckpt.get('model_state_dict', ckpt)
# Instantiate with the repo's expected args (num_classes from vocab):
from src.utils import load_json, p as P
num_classes = len(load_json(P(cfg, 'manifests') / 'vocab.json')) + 1
print('Inspect SLRModel signature, then instantiate. num_classes =', num_classes)

In [ ]:
# Once instantiated & loaded, expose the frozen 2D frontend:
#   frontend = full_model.conv2d   (ResNet18 trunk -> per-frame 512-d)
# frontend.eval(); [p.requires_grad_(False) for p in frontend.parameters()]
# Keep a reference for the dump cell below.
print('Set `frontend` to CorrNet conv2d trunk before continuing.')

## Frame loading + normalization

In [ ]:
import numpy as np, glob, cv2
from src.vocab import frame_glob

CROP = cfg['rgb_dump']['crop']
def load_clip(split, folder):
    paths = sorted(glob.glob(frame_glob(cfg, split, folder)))
    assert paths, f'no frames for {folder}'
    out = []
    for fp in paths:
        img = cv2.cvtColor(cv2.imread(fp), cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        t, l = (h - CROP) // 2, (w - CROP) // 2
        img = img[t:t+CROP, l:l+CROP]
        out.append(img)
    x = np.stack(out).astype(np.float32)          # [T,H,W,3]
    if cfg['rgb_dump']['normalize'] == 'vac_pm1':
        x = x / 127.5 - 1.0                        # ADAPT_HERE if transform differs
    return torch.from_numpy(x).permute(0, 3, 1, 2) # [T,3,H,W]

## Resumable dump loop

In [ ]:
from pathlib import Path
from src.utils import load_json, p, update_json
import torch

CHUNK = cfg['rgb_dump']['chunk']
for split in cfg['dataset']['splits']:
    items = load_json(p(cfg, 'manifests') / f'{split}.json')
    out_dir = p(cfg, 'features_rgb') / split; out_dir.mkdir(parents=True, exist_ok=True)
    fail_log = p(cfg, 'features_rgb') / f'failures_{split}.json'
    done = {f.stem for f in out_dir.glob('*.npy')}
    print(f'{split}: {len(items)} total, {len(done)} already done')
    for n, it in enumerate(items):
        if it['id'] in done:
            continue
        try:
            clip = load_clip(split, it['folder']).to(device)
            feats = []
            with torch.no_grad():
                for i in range(0, len(clip), CHUNK):
                    f = frontend(clip[i:i+CHUNK])        # [chunk,512] expected
                    feats.append(f.float().cpu())
            arr = torch.cat(feats).numpy().astype('float16')
            np.save(out_dir / f"{it['id']}.npy", arr)
            if n % 100 == 0:
                print(f'  {split} {n}/{len(items)}  T={arr.shape[0]} D={arr.shape[1]}')
        except Exception as e:
            print('  FAIL', it['id'], repr(e))
            update_json(fail_log, {it['id']: repr(e)})
    print(f'{split}: done, failures -> {fail_log}')

**Verify dump**

In [ ]:
import numpy as np
from src.utils import load_json, p
for split in cfg['dataset']['splits']:
    items = load_json(p(cfg, 'manifests') / f'{split}.json')
    out_dir = p(cfg, 'features_rgb') / split
    n_npy = len(list(out_dir.glob('*.npy')))
    sample = np.load(next(out_dir.glob('*.npy')))
    print(f'{split}: {n_npy}/{len(items)} dumped, sample shape {sample.shape}, dtype {sample.dtype}')